# Flausch detection -- Data Generation

## Dataset

### Load Dataset and prepare

In [1]:
# Load original data
import pandas as pd
import os
import sys
import numpy as np
path = "Input/Data/train/"
#path = ""
file = "comments.csv"
data = pd.read_csv(path + file, encoding='utf-8')
task1 = pd.read_csv(path + "task1.csv", encoding='utf-8')
task2 = pd.read_csv(path + "task2.csv", encoding='utf-8')

In [5]:
# If "comments_extended.csv" is available use this instead

# comments and task1 are original datasets. They are merged to create data.
# if extended data is available, then use that instead.
# For generating translations, spelling corrections and sentiments see below (takes a lot of time)

# Another option is using "translated_data.csv" which includes only translations

if os.path.exists(path + "comments_extended.csv"):
    data = pd.read_csv(path + "comments_extended.csv", encoding='utf-8')
else:
    # Merge comments and task1 datasets
    data = pd.merge(data, task1, on=["comment_id","document"], how = "outer")

data2 = pd.merge(task2,data, on=["comment_id","document"], how = "left")

print(data.shape, data2.shape, task1.shape, task2.shape)
data.columns, data2.columns

(37057, 16) (15799, 19) (37057, 3) (15799, 5)


(Index(['document', 'comment_id', 'comment', 'flausch', 'translated',
        'spelling_corrected', 'sentiment_orig', 'sentiment_spelling_corrected',
        'sentiment_translated', 'sentiment_anger', 'sentiment_disgust',
        'sentiment_fear', 'sentiment_joy', 'sentiment_neutral',
        'sentiment_sadness', 'sentiment_surprise'],
       dtype='object'),
 Index(['document', 'comment_id', 'type', 'start', 'end', 'comment', 'flausch',
        'translated', 'spelling_corrected', 'sentiment_orig',
        'sentiment_spelling_corrected', 'sentiment_translated',
        'sentiment_anger', 'sentiment_disgust', 'sentiment_fear',
        'sentiment_joy', 'sentiment_neutral', 'sentiment_sadness',
        'sentiment_surprise'],
       dtype='object'))

In [6]:
# add span-column to data2. span contains the text span that is annotated with flausch type
data2["span"] = data2.apply(lambda row: row["comment"][row["start"]:(row["end"]+1)], axis=1)

In [7]:
data2.head(3)

,document,comment_id,type,start,end,comment,flausch,translated,spelling_corrected,sentiment_orig,sentiment_spelling_corrected,sentiment_translated,sentiment_anger,sentiment_disgust,sentiment_fear,sentiment_joy,sentiment_neutral,sentiment_sadness,sentiment_surprise,span
0,NDY-003,1,compliment,0,11,Respekt : o zu mir passt das heut vollkommen :...,yes,Respect : o to me this fits perfectly today :(...,Respekt - ob zu mir passt das heute vollkommen...,0.0,0.0,0.137500,0.317590,0.117705,0.004799,0.272843,0.234663,0.040224,0.012175,Respekt : o
1,NDY-003,1,compliment,48,71,Respekt : o zu mir passt das heut vollkommen :...,yes,Respect : o to me this fits perfectly today :(...,Respekt - ob zu mir passt das heute vollkommen...,0.0,0.0,0.137500,0.317590,0.117705,0.004799,0.272843,0.234663,0.040224,0.012175,Aber Respekt an euch ;)
2,NDY-003,2,positive feedback,0,12,haha geil :D aber ich hab mich am anfang etwas...,yes,haha horny :D but I got a little scared at the...,Haha geil -Da aber ich hab mich am Anfang etwa...,0.7,0.7,0.503125,0.002527,0.001026,0.978153,0.002605,0.004189,0.009158,0.002342,haha geil :D


In [8]:
# function to get all spans for a comment.
# Returns list of spans, list of start-end-pairs and list of flausch types for a given document and comment id.
def get_all_spans(doc,id,comment,task2):
    doc_bool = np.array(task2["document"] == doc)
    id_bool = np.array(task2["comment_id"] == id)
    bool = doc_bool & id_bool
    sub = task2[bool].reset_index()
    spans = []
    span_pairs = []
    types = []
    for i in range(len(sub)):
        s = sub["start"][i]
        e = sub["end"][i]
        spans.append(comment[s:e])
        span_pairs.append((s,e))
        types.append(sub["type"][i])
    return spans, span_pairs, types

In [9]:
# adds columns spans, span-pairs and types to data
# values are lists that are empty if flausch == no
data["spans"] = data.apply(lambda x: get_all_spans(x["document"], x["comment_id"], x["comment"],task2),axis=1)
data["span_pairs"] = data["spans"].apply(lambda x: x[1])
data["types"] = data["spans"].apply(lambda x: x[2])
data["spans"] = data["spans"].apply(lambda x: x[0])

In [10]:
data[data["flausch"]=="yes"].head(3)

,document,comment_id,comment,flausch,translated,spelling_corrected,sentiment_orig,sentiment_spelling_corrected,sentiment_translated,sentiment_anger,sentiment_disgust,sentiment_fear,sentiment_joy,sentiment_neutral,sentiment_sadness,sentiment_surprise,spans,span_pairs,types
0,NDY-003,1,Respekt : o zu mir passt das heut vollkommen :...,yes,Respect : o to me this fits perfectly today :(...,Respekt - ob zu mir passt das heute vollkommen...,0.0,0.0,0.137500,0.317590,0.117705,0.004799,0.272843,0.234663,0.040224,0.012175,"[Respekt : o, Aber Respekt an euch ;)]","[(0, 11), (48, 71)]","[compliment, compliment]"
1,NDY-003,2,haha geil :D aber ich hab mich am anfang etwas...,yes,haha horny :D but I got a little scared at the...,Haha geil -Da aber ich hab mich am Anfang etwa...,0.7,0.7,0.503125,0.002527,0.001026,0.978153,0.002605,0.004189,0.009158,0.002342,[haha geil :D],"[(0, 12)]",[positive feedback]
2,NDY-003,3,würd gern mit euch tanzen * - * haha,yes,Would like to dance with you * - * haha,"Würde gern mit euch tanzen, -- -, -- haha!",0.0,0.0,0.200000,0.003334,0.002170,0.000931,0.900905,0.061671,0.009552,0.021437,[würd gern mit euch tanzen * - *],"[(0, 31)]",[positive feedback]


### Split in test and train data

In [11]:
# add id column
data["id"] =    data["document"] + "_" + data["comment_id"].astype(str)
data2["id"] = data2["document"] + "_" + data2["comment_id"].astype(str)

In [12]:
data.head(3)

,document,comment_id,comment,flausch,translated,spelling_corrected,sentiment_orig,sentiment_spelling_corrected,sentiment_translated,sentiment_anger,sentiment_disgust,sentiment_fear,sentiment_joy,sentiment_neutral,sentiment_sadness,sentiment_surprise,spans,span_pairs,types,id
0,NDY-003,1,Respekt : o zu mir passt das heut vollkommen :...,yes,Respect : o to me this fits perfectly today :(...,Respekt - ob zu mir passt das heute vollkommen...,0.0,0.0,0.137500,0.317590,0.117705,0.004799,0.272843,0.234663,0.040224,0.012175,"[Respekt : o, Aber Respekt an euch ;)]","[(0, 11), (48, 71)]","[compliment, compliment]",NDY-003_1
1,NDY-003,2,haha geil :D aber ich hab mich am anfang etwas...,yes,haha horny :D but I got a little scared at the...,Haha geil -Da aber ich hab mich am Anfang etwa...,0.7,0.7,0.503125,0.002527,0.001026,0.978153,0.002605,0.004189,0.009158,0.002342,[haha geil :D],"[(0, 12)]",[positive feedback],NDY-003_2
2,NDY-003,3,würd gern mit euch tanzen * - * haha,yes,Would like to dance with you * - * haha,"Würde gern mit euch tanzen, -- -, -- haha!",0.0,0.0,0.200000,0.003334,0.002170,0.000931,0.900905,0.061671,0.009552,0.021437,[würd gern mit euch tanzen * - *],"[(0, 31)]",[positive feedback],NDY-003_3


In [13]:
# split data into train and test sets (test set is not touched for training or hyperparameter tuning)
from sklearn.model_selection import train_test_split

train_task1, test_task1 = train_test_split(data, test_size=0.1, random_state=42)

In [14]:
train_task2 = data2[data2["id"].isin(train_task1["id"])]
test_task2 = data2[data2["id"].isin(test_task1["id"])]

In [15]:
train_task2.head(3)

,document,comment_id,type,start,end,comment,flausch,translated,spelling_corrected,sentiment_orig,...,sentiment_translated,sentiment_anger,sentiment_disgust,sentiment_fear,sentiment_joy,sentiment_neutral,sentiment_sadness,sentiment_surprise,span,id
0,NDY-003,1,compliment,0,11,Respekt : o zu mir passt das heut vollkommen :...,yes,Respect : o to me this fits perfectly today :(...,Respekt - ob zu mir passt das heute vollkommen...,0.0,...,0.137500,0.317590,0.117705,0.004799,0.272843,0.234663,0.040224,0.012175,Respekt : o,NDY-003_1
1,NDY-003,1,compliment,48,71,Respekt : o zu mir passt das heut vollkommen :...,yes,Respect : o to me this fits perfectly today :(...,Respekt - ob zu mir passt das heute vollkommen...,0.0,...,0.137500,0.317590,0.117705,0.004799,0.272843,0.234663,0.040224,0.012175,Aber Respekt an euch ;),NDY-003_1
2,NDY-003,2,positive feedback,0,12,haha geil :D aber ich hab mich am anfang etwas...,yes,haha horny :D but I got a little scared at the...,Haha geil -Da aber ich hab mich am Anfang etwa...,0.7,...,0.503125,0.002527,0.001026,0.978153,0.002605,0.004189,0.009158,0.002342,haha geil :D,NDY-003_2


In [ ]:
# for safety reasons commented out to not overwrite existing files
#train_task1.to_json(path + "train_task1.json")
#test_task1.to_json(path + "test_task1.json")
#train_task2.to_json(path + "train_task2.json")
#test_task2.to_json(path + "test_task2.json")

## Generate Translations

### Model: "Helsinki-NLP/opus-mt-de-en"

Runs on local machine. (690 minutes)

In [ ]:
#!pip install --upgrade translators

In [ ]:
import pandas as pd
from transformers import pipeline
import time
import os

# --- 2. Konfiguration ---
MODEL_NAME = "Helsinki-NLP/opus-mt-de-en"
BATCH_SIZE = 64  # Optimale Batch-Größe kann je nach GPU/CPU variieren
SAVE_INTERVAL = 500 # Speichert alle 500 übersetzten Kommentare
OUTPUT_FILE = "translated_data_checkpoint.csv" # Dateiname für Zwischenspeicherung und Endprodukt
BACKUP_DIR = "translation_backups" # Verzeichnis für Backups vor dem Überschreiben

# Sicherstellen, dass das Backup-Verzeichnis existiert
os.makedirs(BACKUP_DIR, exist_ok=True)

# --- 3. Übersetzer laden ---
print(f"Lade Übersetzungsmodell: {MODEL_NAME}...")
# Das Pipeline erkennt automatisch den passenden Tokenizer und das Modell.
# Falls du eine GPU hast, kannst du device=0 hinzufügen:
# translator = pipeline("translation", model=MODEL_NAME, device=0)
translator = pipeline("translation", model=MODEL_NAME)
print("Modell erfolgreich geladen.")

# --- 4. Übersetzungsfunktion ---
def translate_comments_batched(dataframe, comment_column="comment", batch_size=BATCH_SIZE, save_interval=SAVE_INTERVAL, output_file=OUTPUT_FILE, backup_dir=BACKUP_DIR):
    total_comments = len(dataframe)

    # Prüfen, ob eine Teildatei existiert und Laden des Fortschritts
    if os.path.exists(output_file):
        print(f"Bestehende Übersetzungsdatei gefunden: {output_file}. Lade Fortschritt...")
        try:
            temp_df = pd.read_csv(output_file)
            if "translated" in temp_df.columns:
                # Sicherstellen, dass die Längen übereinstimmen
                if len(temp_df) == total_comments:
                    # Wenn die Datei die gleiche Größe hat und eine übersetzte Spalte,
                    # gehen wir davon aus, dass alles übersetzt ist.
                    print("Alle Kommentare scheinen bereits übersetzt zu sein. Beende Vorgang.")
                    return temp_df

                # Finde den letzten übersetzten Index
                last_translated_idx = temp_df["translated"].last_valid_index()
                if last_translated_idx is not None:
                    # Wenn der letzte Wert None ist, aber es davor übersetzte gab,
                    # müssen wir den Index des letzten *echten* übersetzten Eintrags finden.
                    start_index = last_translated_idx + 1
                    # Wenn die letzten Einträge leer sind, suchen wir nach dem letzten Nicht-Null-Wert
                    while start_index > 0 and pd.isna(temp_df.loc[start_index - 1, "translated"]):
                        start_index -= 1

                    if start_index < total_comments:
                        dataframe["translated"] = temp_df["translated"] # Vorhandene Übersetzungen übernehmen
                        print(f"Fortsetzung der Übersetzung ab Kommentar {start_index} von {total_comments}.")
                    else:
                        print("Alle Kommentare scheinen bereits übersetzt zu sein (nachprüfung). Beende Vorgang.")
                        return temp_df
                else: # Keine übersetzten Kommentare in der Datei gefunden, starte von vorn
                    dataframe["translated"] = pd.Series([None] * total_comments) # Initialisiere mit None
                    start_index = 0
                    print("Starte Übersetzung von Anfang an (keine gültigen Übersetzungen in der Datei gefunden).")
            else: # Datei existiert, hat aber keine "translated"-Spalte
                dataframe["translated"] = pd.Series([None] * total_comments)
                start_index = 0
                print("Starte Übersetzung von Anfang an (keine 'translated' Spalte gefunden).")
        except Exception as e:
            print(f"Fehler beim Laden der Checkpoint-Datei: {e}. Starte von vorne.")
            dataframe["translated"] = pd.Series([None] * total_comments) # Initialisiere mit None
            start_index = 0
    else:
        dataframe["translated"] = pd.Series([None] * total_comments) # Initialisiere mit None
        start_index = 0
        print("Starte Übersetzung von Anfang an (keine Checkpoint-Datei gefunden).")

    start_time = time.time()

    for i in range(start_index, total_comments, batch_size):
        batch_end_index = min(i + batch_size, total_comments)

        # Sicherstellen, dass der Batch nur die noch nicht übersetzten Texte enthält
        batch_to_translate_df = dataframe.iloc[i:batch_end_index]

        # Filtere leere Strings oder NaN-Werte, um sie nicht an das Modell zu senden
        # und speichere die ursprünglichen Indizes, um die Ergebnisse richtig zuzuordnen
        texts_to_process_indices = batch_to_translate_df[comment_column].apply(lambda x: pd.notna(x) and str(x).strip() != "").index.tolist()
        texts_to_process = [str(dataframe.loc[idx, comment_column]) for idx in texts_to_process_indices]

        translated_batch = []
        if texts_to_process:
            try:
                # Das Pipeline kann eine Liste von Strings verarbeiten
                results = translator(texts_to_process)
                translated_batch = [item['translation_text'] for item in results]
            except Exception as e:
                print(f"Fehler beim Übersetzen des Batches {i}-{batch_end_index-1}: {e}")
                # Im Fehlerfall füllen wir den Batch mit None oder dem Originaltext
                translated_batch = [None] * len(texts_to_process)

        # Ergebnisse den ursprünglichen Indizes zuordnen
        translated_results_map = {}
        for j, original_idx in enumerate(texts_to_process_indices):
            if j < len(translated_batch):
                translated_results_map[original_idx] = translated_batch[j]
            else:
                translated_results_map[original_idx] = None # Fehler oder fehlende Übersetzung

        # Original-DataFrame aktualisieren
        for idx in range(i, batch_end_index):
            if idx in translated_results_map:
                dataframe.loc[idx, "translated"] = translated_results_map[idx]
            elif pd.isna(dataframe.loc[idx, comment_column]) or str(dataframe.loc[idx, comment_column]).strip() == "":
                dataframe.loc[idx, "translated"] = "" # Leere Eingaben bleiben leer
            else:
                dataframe.loc[idx, "translated"] = None # Oder den Originaltext lassen, wenn ein unerwarteter Fehler auftritt

        current_progress = batch_end_index
        if current_progress % save_interval == 0 or current_progress == total_comments:
            elapsed_time = time.time() - start_time
            print(f"Fortschritt: {current_progress}/{total_comments} Posts übersetzt. "
                  f"Verstrichene Zeit: {elapsed_time:.2f} Sekunden.")

            # Sicherungskopie erstellen, bevor die Hauptdatei überschrieben wird
            backup_path = os.path.join(backup_dir, f"{OUTPUT_FILE.replace('.csv', '')}_backup_{current_progress}.csv")
            dataframe.to_csv(backup_path, index=False)
            print(f"Zwischenstand als Backup gespeichert: {backup_path}")

            # Hauptdatei aktualisieren
            dataframe.to_csv(output_file, index=False)
            print(f"Zwischenstand in {output_file} gespeichert.")

    print("\nÜbersetzung abgeschlossen!")
    return dataframe

# --- 5. Ausführung ---
if __name__ == "__main__":
    # Stelle sicher, dass dein 'data' DataFrame hier geladen ist,
    # wenn es nicht das Beispiel-DataFrame ist.
    # z.B.: data = pd.read_csv("your_data.csv")

    # Füge eine leere Spalte "translated" hinzu, falls sie noch nicht existiert
    if "translated" not in data.columns:
        data["translated"] = pd.Series([None] * len(data))

    final_data = translate_comments_batched(data.copy()) # Erstelle eine Kopie, um das Original nicht direkt zu ändern

    print("\nÜbersetzter DataFrame:")
    print(final_data.head())

    # Optional: Speichere das finale Ergebnis erneut
    final_data.to_csv("final_translated_data.csv", index=False)
    print(f"Endgültige übersetzte Daten in 'final_translated_data.csv' gespeichert.")

Lade Übersetzungsmodell: Helsinki-NLP/opus-mt-de-en...


Device set to use cpu


Modell erfolgreich geladen.
Starte Übersetzung von Anfang an (keine Checkpoint-Datei gefunden).
Fortschritt: 8000/37057 Posts übersetzt. Verstrichene Zeit: 8493.89 Sekunden.
Zwischenstand als Backup gespeichert: translation_backups\translated_data_checkpoint_backup_8000.csv
Zwischenstand in translated_data_checkpoint.csv gespeichert.
Fortschritt: 16000/37057 Posts übersetzt. Verstrichene Zeit: 18746.84 Sekunden.
Zwischenstand als Backup gespeichert: translation_backups\translated_data_checkpoint_backup_16000.csv
Zwischenstand in translated_data_checkpoint.csv gespeichert.


Token indices sequence length is longer than the specified maximum sequence length for this model (1516 > 512). Running this sequence through the model will result in indexing errors
Your input_length: 1516 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Fehler beim Übersetzen des Batches 16576-16639: index out of range in self


...

Fehler beim Übersetzen des Batches 19712-19775: index out of range in self
Fortschritt: 24000/37057 Posts übersetzt. Verstrichene Zeit: 28602.66 Sekunden.
Zwischenstand als Backup gespeichert: translation_backups\translated_data_checkpoint_backup_24000.csv
Zwischenstand in translated_data_checkpoint.csv gespeichert.


Your input_length: 538 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Fehler beim Übersetzen des Batches 27520-27583: index out of range in self
Fortschritt: 32000/37057 Posts übersetzt. Verstrichene Zeit: 36039.61 Sekunden.
Zwischenstand als Backup gespeichert: translation_backups\translated_data_checkpoint_backup_32000.csv
Zwischenstand in translated_data_checkpoint.csv gespeichert.


Your input_length: 2671 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


Fehler beim Übersetzen des Batches 32128-32191: index out of range in self
Fortschritt: 37057/37057 Posts übersetzt. Verstrichene Zeit: 41147.23 Sekunden.
Zwischenstand als Backup gespeichert: translation_backups\translated_data_checkpoint_backup_37057.csv
Zwischenstand in translated_data_checkpoint.csv gespeichert.

Übersetzung abgeschlossen!

Übersetzter DataFrame:
  document  comment_id                                            comment  \
0  NDY-003           1  Respekt : o zu mir passt das heut vollkommen :...   
1  NDY-003           2  haha geil :D aber ich hab mich am anfang etwas...   
2  NDY-003           3               würd gern mit euch tanzen * - * haha   
3  NDY-003           4  Einfach nur wieder geil ... ich mag eure texte...   
4  NDY-003           5                                 das ist so geil :)   

  flausch                                         translated  
0     yes  Respect : o to me this fits perfectly today :(...  
1     yes  haha horny :D but I got a litt

In [ ]:
missing_translation = final_data[final_data["translated"].isna()].copy()
for i in range(len(missing_translation)):
    text = missing_translation["comment"].iloc[i]
    translated = translator(text, src_lang="de", tgt_lang="en", max_length= 512, truncation= True)[0]["translation_text"]
    missing_translation["translated"].iloc[i] = translated

C:\Users\Wiebke Petersen\AppData\Local\Temp\ipykernel_38764\1026092389.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_translation["translated"].iloc[i] = translated
Your input_length: 512 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 512 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 512 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 512 is bigger than 0.9 * max_length: 512. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_

In [ ]:
for i in missing_translation.index:
    final_data.loc[i,"translated"] = missing_translation.loc[i,"translated"]

In [ ]:
final_data.to_csv(path + "translated_data.csv", index=False)

## Generate spelling correction

### Model: "oliverguhr/spelling-correction-german-base"

In [33]:
import pandas as pd
import pickle

In [34]:
# adding to the data which includes translations
#comments = final_data
comments = pd.read_csv(path + "translated_data.csv")

# adding to the original data, no translations
#comments = pd.read_csv(path + "comments.csv")

In [35]:
# If this doesn't work, comments_fixed.pkl needs to be generated
# using the pipeline for spelling correction below.
# (Not recommenended because it takes a lot of time)

with open(path + "comments_fixed.pkl", "rb") as infile: 
   comments_fixed = pickle.load(infile)


# save as comments_extended.csv with spelling corrections added

comments["spelling_corrected"] = comments_fixed
comments.to_csv(path + "comments_extended.csv", index=False)

In [ ]:
from transformers import pipeline

fix_spelling = pipeline("text2text-generation",model="oliverguhr/spelling-correction-german-base")

In [ ]:
# generating a versin with corrected spelling for each comment

comments_fixed = []

for i in range(len(comments)):
    
    comment = comments.iloc[i].item()
    comment_fixed = fix_spelling("correct: " + comment, max_length=256)[0]["generated_text"]
    comments_fixed.append(comment_fixed)


# saving the spelling corrected versions of the comments
with open(path + "comments_fixed.pkl", "wb") as outfile: 
   pickle.dump(comments_fixed, outfile)

## Generate sentiments

### Polarity

In [ ]:
import pandas as pd
import pickle

In [ ]:
# adding to the data which includes translated and spelling corrected versions
comments = pd.read_csv(path + "comments_extended.csv")

# adding to the original data, no translations or spelling correction
#comments = pd.read_csv(path + "comments.csv")

In [37]:
# If the pickle files don't exist, need to run code below.

with open(path + "sentiments_orig.pkl", "rb") as infile: 
   sentiments_orig = pickle.load(infile)

with open(path + "sentiments_spelling_corrected.pkl", "rb") as infile: 
   sentiments_spelling_corrected = pickle.load(infile)

with open(path + "sentiments_translated.pkl", "rb") as infile: 
   sentiments_translated = pickle.load(infile)


# save to comments_extended.csv with sentiment scores added

comments["sentiment_orig"] = sentiments_orig
comments["sentiment_spelling_corrected"] = sentiments_spelling_corrected
comments["sentiment_translated"] = sentiments_translated

comments.to_csv(path + "comments_extended.csv", index=False)

In [ ]:
#!pip install textblob==0.15.2
from textblob_de import TextBlobDE
from textblob import TextBlob

In [ ]:
# get sentiment polarity (from 1.0 (most positive) to -1.0 (most negative))
# for comments in original, spelling corrected and translated versions.
# Spelling corrected and translated require the respective columns
# thus need to be commented out if using the original data.

sentiments_orig = []
sentiments_spelling_corrected = []
sentiments_translated = []

for i in range(len(comments)):

    comment = comments.iloc[i]["comment"]
    blob = TextBlobDE(comment)
    sentiments_orig.append(blob.sentiment.polarity)

    spelling_corrected = comments.iloc[i]["spelling_corrected"]
    blob = TextBlobDE(spelling_corrected)
    sentiments_spelling_corrected.append(blob.sentiment.polarity)

    translated = comments.iloc[i]["translated"]
    blob = TextBlob(str(translated)) # some translations don't exist (i: 6920, 11903, 17510)
    sentiments_translated.append(blob.sentiment.polarity)


# save the sentiment polarities as separate pickle files

with open(path + "sentiments_orig.pkl", "wb") as outfile: 
   pickle.dump(sentiments_orig, outfile)

with open(path + "sentiments_spelling_corrected.pkl", "wb") as outfile: 
   pickle.dump(sentiments_spelling_corrected, outfile)

with open(path + "sentiments_translated.pkl", "wb") as outfile: 
   pickle.dump(sentiments_translated, outfile)

### Seven sentiments

Model: "j-hartmann/emotion-english-distilroberta-base"

In [2]:
import pandas as pd
import pickle

In [3]:
# adding to the data which includes translated and spelling corrected versions
# and sentiment polarity for all three versions
# requires at least the column with the translated comments
comments = pd.read_csv(path + "comments_extended.csv")

In [4]:
with open(path + "sentiments_anger.pkl", "rb") as infile: 
   sentiments_anger = pickle.load(infile)

with open(path + "sentiments_disgust.pkl", "rb") as infile: 
   sentiments_disgust = pickle.load(infile)

with open(path + "sentiments_fear.pkl", "rb") as infile: 
   sentiments_fear = pickle.load(infile)

with open(path + "sentiments_joy.pkl", "rb") as infile: 
   sentiments_joy = pickle.load(infile)

with open(path + "sentiments_neutral.pkl", "rb") as infile: 
   sentiments_neutral = pickle.load(infile)

with open(path + "sentiments_sadness.pkl", "rb") as infile: 
   sentiments_sadness = pickle.load(infile)

with open(path + "sentiments_surprise.pkl", "rb") as infile: 
   sentiments_surprise = pickle.load(infile)

# save to comments_extended.csv with sentiment scores added

comments["sentiment_anger"] = sentiments_anger
comments["sentiment_disgust"] = sentiments_disgust
comments["sentiment_fear"] = sentiments_fear
comments["sentiment_joy"] = sentiments_joy
comments["sentiment_neutral"] = sentiments_neutral
comments["sentiment_sadness"] = sentiments_sadness
comments["sentiment_surprise"] = sentiments_surprise

comments.to_csv(path + "comments_extended.csv", index=False)

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", truncation=True, max_length=512, return_all_scores=True)

In [ ]:
# get scores (0.0 - 1.0) for seven different emotions
# for the English, translated comments

sentiments_anger = []
sentiments_disgust = []
sentiments_fear = []
sentiments_joy = []
sentiments_neutral = []
sentiments_sadness = []
sentiments_surprise = []

for i in range(len(comments)):
    emotions = classifier(str(comments.iloc[i]["translated"]))[0]

    sentiments_anger.append(emotions[0]["score"])
    sentiments_disgust.append(emotions[1]["score"])
    sentiments_fear.append(emotions[2]["score"])
    sentiments_joy.append(emotions[3]["score"])
    sentiments_neutral.append(emotions[4]["score"])
    sentiments_sadness.append(emotions[5]["score"])
    sentiments_surprise.append(emotions[6]["score"])

# save lists for each sentimenr
with open(path + "sentiments_anger.pkl", "wb") as outfile: 
   pickle.dump(sentiments_anger, outfile)
 
with open(path + "sentiments_disgust.pkl", "wb") as outfile: 
   pickle.dump(sentiments_disgust, outfile)

with open(path + "sentiments_fear.pkl", "wb") as outfile: 
   pickle.dump(sentiments_fear, outfile)

with open(path + "sentiments_joy.pkl", "wb") as outfile: 
   pickle.dump(sentiments_joy, outfile)

with open(path + "sentiments_neutral.pkl", "wb") as outfile: 
   pickle.dump(sentiments_neutral, outfile)

with open(path + "sentiments_sadness.pkl", "wb") as outfile: 
   pickle.dump(sentiments_sadness, outfile)

with open(path + "sentiments_surprise.pkl", "wb") as outfile: 
   pickle.dump(sentiments_surprise, outfile)